# Phase 2: CHIVA Task-Specific Fine-Tuning
## Qwen2.5-7B with LoRA on Medical Classification Tasks

Run each cell sequentially to train the model on CHIVA shunt classification and ligation tasks.

**Requirements:**
```bash
pip install torch transformers peft jsonlines matplotlib
```

## Cell 1: Setup & Imports

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

import torch
import json
import jsonlines
from pathlib import Path
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

print("✓ Imports successful")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2: Configure Paths
⚠️ **IMPORTANT**: Modify these paths to match your Ubuntu setup

In [ ]:
# MODIFY THESE FOR YOUR SETUP
BASE_MODEL = "Qwen/Qwen2.5-7B"

# Example: /home/username/llm_finetuning
BASE_DIR = "/home/username/llm_finetuning"  # CHANGE THIS

DATA_DIR = f"{BASE_DIR}/latest_data"
DATASET_FILE = f"{DATA_DIR}/training_data_FRESH.jsonl"
OUTPUT_DIR = f"{BASE_DIR}/qwen_chiva_tasks_lora"
CACHE_DIR = f"{BASE_DIR}/.cache"

# Create directories
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Verify dataset exists
if not os.path.exists(DATASET_FILE):
    print(f"⚠️ ERROR: {DATASET_FILE} not found!")
    print("Please check your BASE_DIR path")
else:
    with jsonlines.open(DATASET_FILE) as f:
        count = sum(1 for _ in f)
    print(f"✓ Dataset found: {count} examples")
    print(f"✓ Output dir: {OUTPUT_DIR}")

## Cell 3: Dataset Class

In [ ]:
class SimpleDataset(Dataset):
    """Load JSONL training data with input/output pairs."""

    def __init__(self, jsonl_file, tokenizer, max_len=512):
        self.data = []
        with jsonlines.open(jsonl_file) as f:
            for line in f:
                self.data.append(line)
        self.tokenizer = tokenizer
        self.max_len = max_len
        print(f"✓ Loaded {len(self.data)} examples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = f"Q: {item['input']}\nA: {item['output']}"

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }

print("✓ Dataset class defined")

## Cell 4: Load Tokenizer & Model
⏱️ This takes 2-3 minutes

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)
tokenizer.pad_token = tokenizer.eos_token
print(f"✓ Tokenizer loaded - Vocab: {len(tokenizer)}")

print("\nLoading model (2-3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded - {total_params / 1e9:.1f}B parameters")

## Cell 5: Apply LoRA

In [ ]:
print("Applying LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Rank
    lora_alpha=32,          # Scaling
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.train()  # Training mode

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"✓ LoRA applied")
print(f"  Trainable: {trainable / 1e6:.1f}M ({100 * trainable / total:.2f}%)")
print(f"  Total: {total / 1e9:.1f}B")

## Cell 6: Prepare Dataset

In [ ]:
print("Loading dataset...")
dataset = SimpleDataset(DATASET_FILE, tokenizer, max_len=512)

batch_size = 2  # Adjust if needed
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"\n✓ Dataset ready")
print(f"  Examples: {len(dataset)}")
print(f"  Batch size: {batch_size}")
print(f"  Total batches: {len(dataloader)}")
print(f"  Effective batch (w/ accumulation): {batch_size * 4}")

## Cell 7: Setup Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01,
)

num_epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f"✓ Optimizer ready")
print(f"  LR: 1e-4")
print(f"  Epochs: {num_epochs}")
print(f"  Scheduler: Cosine Annealing")

## Cell 8: TRAINING LOOP
⏱️ This takes ~15-20 minutes

In [ ]:
print("="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Start: {datetime.now().isoformat()}\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")
model = model.to(device)

total_loss = 0
global_step = 0
losses_per_epoch = []

for epoch in range(num_epochs):
    epoch_loss = 0
    epoch_steps = 0

    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 40)

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )
        loss = outputs.loss

        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Step (every 4 batches for gradient accumulation)
        if (batch_idx + 1) % 4 == 0:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item()
        total_loss += loss.item()
        epoch_steps += 1
        global_step += 1

        if (batch_idx + 1) % 5 == 0:
            avg = epoch_loss / epoch_steps
            print(f"  Batch {batch_idx+1}/{len(dataloader)}, Loss: {loss.item():.4f}, Avg: {avg:.4f}")

    scheduler.step()
    avg_epoch = epoch_loss / epoch_steps
    losses_per_epoch.append(avg_epoch)
    print(f"✓ Epoch {epoch+1} - Loss: {avg_epoch:.4f}\n")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"End: {datetime.now().isoformat()}")
print(f"Final loss: {total_loss / global_step:.4f}")
print(f"Total steps: {global_step}")

## Cell 9: Save Model

In [ ]:
print("\nSaving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

config = {
    'model': BASE_MODEL,
    'num_examples': len(dataset),
    'epochs': num_epochs,
    'learning_rate': 1e-4,
    'final_loss': total_loss / global_step,
    'total_steps': global_step,
    'timestamp': datetime.now().isoformat(),
}

with open(os.path.join(OUTPUT_DIR, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Model saved to: {OUTPUT_DIR}")
print(f"\nFiles created:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  - {f}")

## Cell 10: Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(losses_per_epoch) + 1), losses_per_epoch, marker='o', linestyle='-', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.title('CHIVA Model Training Loss', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_loss.png", dpi=100)
plt.show()

print(f"✓ Plot saved")

## Cell 11: Test the Model

In [ ]:
print("\n" + "="*80)
print("TESTING FINE-TUNED MODEL")
print("="*80 + "\n")

test_cases = [
    "Classify: EP N1->N2 at y=0.06 with RP N2->N1 at y=0.25. No N3.",
    "For TYPE 1 shunt, what is the ligation strategy?",
    "Classify: EP N2->N3 at y=0.20 with no EP N1->N2.",
    "What is the difference between TYPE 2B and TYPE 2C?",
]

model.eval()
with torch.no_grad():
    for i, prompt in enumerate(test_cases, 1):
        print(f"[Test {i}]")
        print(f"Q: {prompt}")

        inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            inputs,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            do_sample=False,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response[len(prompt):].strip()

        print(f"A: {response}\n")

print("✓ Testing complete")

## Cell 12: Summary & Next Steps

In [ ]:
print("="*80)
print("PHASE 2 TRAINING COMPLETE!")
print("="*80)
print(f"\nModel Location: {OUTPUT_DIR}")
print(f"\n📊 Training Results:")
print(f"  - Epochs: {num_epochs}")
print(f"  - Training examples: {len(dataset)}")
print(f"  - Final loss: {total_loss / global_step:.4f}")
print(f"  - Total steps: {global_step}")
print(f"\n💾 Files saved:")
print(f"  - adapter_model.bin (LoRA weights)")
print(f"  - adapter_config.json")
print(f"  - tokenizer files")
print(f"  - training_config.json")
print(f"  - training_loss.png (plot)")
print(f"\n🚀 To use the model:")
print(f"""
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = "Qwen/Qwen2.5-7B"
lora_path = "{OUTPUT_DIR}"

model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, lora_path)
tokenizer = AutoTokenizer.from_pretrained(lora_path)

model.eval()
with torch.no_grad():
    outputs = model.generate(tokenizer.encode("...", return_tensors="pt").to(model.device))
    print(tokenizer.decode(outputs[0]))
""")
print("\n📝 Next steps:")
print("  1. Copy model to deployment server")
print("  2. Test with real patient cases")
print("  3. Integrate with clinical workflow")